# J4-PM — Transformers & Hugging Face

Dans ce notebook on continue sur **SST-2** (la même tâche qu'en J4-AM) pour mesurer le gain apporté par les embeddings contextuels.

| Partie | Méthode | Attendu |
|--------|---------|----------|
| 1 | Explorer le tokenizer HuggingFace | sous-mots, padding, `attention_mask` |
| 2 | DistilBERT comme extracteur de features | token `[CLS]` + LogReg sklearn |
| 3 | Fine-tuning DistilBERT | `Trainer` end-to-end |
| 4 | Comparaison | GloVe LogReg → BERT frozen → DistilBERT fine-tuné |

In [ ]:
!pip install -q transformers datasets evaluate

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")

## Partie 1 — Explorer le tokenizer HuggingFace

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sous-mots : les mots rares sont découpés en fragments connus
examples = ["unhappiness", "tokenization", "DistilBERT", "cat"]
for word in examples:
    tokens = tokenizer.tokenize(word)
    print(f"{word!r:20s} → {tokens}")

In [ ]:
sentences = [
    "The movie was great!",
    "I absolutely hated this film.",
    "Ok.",
]

batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

print("input_ids      :", batch["input_ids"].shape)   # [3, L]
print("attention_mask :", batch["attention_mask"].shape)
print()
print("input_ids :\n", batch["input_ids"])
print("\nattention_mask (1=vrai token, 0=padding) :\n", batch["attention_mask"])

## Partie 2 — DistilBERT comme extracteur de features

On charge DistilBERT **gelé** (aucun paramètre mis à jour) et on utilise le vecteur du token spécial `[CLS]` — le premier token de chaque séquence — comme représentation de la phrase entière. On entraîne ensuite un simple LogReg par-dessus.

In [ ]:
ds = load_dataset("glue", "sst2")

# Sous-ensemble pour ne pas attendre trop longtemps
N_TRAIN = 5000
train_texts  = ds["train"]["sentence"][:N_TRAIN]
train_labels = ds["train"]["label"][:N_TRAIN]
val_texts    = ds["validation"]["sentence"]
val_labels   = ds["validation"]["label"]

print(f"train : {len(train_texts)} exemples")
print(f"val   : {len(val_texts)} exemples")

In [ ]:
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert.eval()


def extract_cls(texts, tokenizer, model, batch_size=64):
    """Retourne un tableau (N, d) avec le vecteur [CLS] de chaque phrase."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i : i + batch_size],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls)
    return np.vstack(all_embeddings)


print("Extraction train…")
X_train = extract_cls(train_texts, tokenizer, bert)
print("Extraction val…")
X_val = extract_cls(val_texts, tokenizer, bert)
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_labels)
acc_frozen = accuracy_score(val_labels, clf.predict(X_val))
print(f"DistilBERT frozen + LogReg : {acc_frozen:.2%}")

## Partie 3 — Fine-tuning DistilBERT

On laisse maintenant tous les paramètres se mettre à jour sur SST-2. Le `Trainer` gère la boucle d'entraînement, le padding dynamique, les checkpoints et l'évaluation.

In [ ]:
raw_train = ds["train"].select(range(N_TRAIN))
raw_val   = ds["validation"]


def tokenize_dataset(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)


tok_train = raw_train.map(tokenize_dataset, batched=True)
tok_val   = raw_val.map(tokenize_dataset, batched=True)

# Le Trainer attend une colonne "labels" (et non "label")
tok_train = tok_train.rename_column("label", "labels")
tok_val   = tok_val.rename_column("label", "labels")
tok_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tok_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [ ]:
trainer.train()

In [ ]:
preds_output = trainer.predict(tok_val)
y_pred = preds_output.predictions.argmax(axis=1)
acc_ft = accuracy_score(val_labels, y_pred)
print(f"DistilBERT fine-tuné : {acc_ft:.2%}")

## Partie 4 — Comparaison

On résume les accuracies obtenues sur SST-2 tout au long des deux journées.

In [ ]:
# Résultats J4-AM (rappel)
ACC_GLOVE_FROZEN  = 0.799   # GloVe frozen  + MLP  (J4-AM)
ACC_GLOVE_FT      = 0.836   # GloVe fine-tuné + MLP (J4-AM)

results = {
    "GloVe frozen  (J4-AM)": ACC_GLOVE_FROZEN,
    "GloVe fine-tuné (J4-AM)": ACC_GLOVE_FT,
    "DistilBERT frozen (J4-PM)": acc_frozen,
    "DistilBERT fine-tuné (J4-PM)": acc_ft,
}

print(f"{'Méthode':<35} {'Accuracy':>9}")
print("-" * 46)
for name, acc in results.items():
    print(f"{name:<35} {acc:>8.2%}")